# q5: new vs returning
do first-time visitors buy, or do people come back first? do new visitors who drop off return later?
funnel by visitor type (q5) and first visit result vs buying later (q5b) in sql/07_new_vs_returning.sql.


In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)


In [2]:
sql = """
select
    case
        when is_new_visitor then 'new'
        when not is_new_visitor then 'returning'
        else 'unknown' 
    end as visitor_type,
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(added_to_cart or began_checkout or purchased) as reached_cart,
    countif(began_checkout or purchased) as reached_checkout,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
group by visitor_type
order by sessions desc
"""
visitors = client.query(sql).to_dataframe()
visitors

,visitor_type,sessions,reached_view,reached_cart,reached_checkout,purchases,revenue
0,new,261238,52231,11643,5974,1506,102941.0
1,returning,98891,25083,8690,5132,2538,204699.0


In [3]:
visitors["view_rate"] = (visitors["reached_view"] / visitors["sessions"]).round(4)
visitors["cart_rate"] = (visitors["reached_cart"] / visitors["reached_view"]).round(4)
visitors["checkout_rate"] = (visitors["reached_checkout"] / visitors["reached_cart"]).round(4)
visitors["purchase_rate"] = (visitors["purchases"] / visitors["reached_checkout"]).round(4)
visitors["conversion_rate"] = (visitors["purchases"] / visitors["sessions"]).round(4)
visitors["revenue_per_session"] = (visitors["revenue"] / visitors["sessions"]).round(2)
visitors["avg_order_value"] = (visitors["revenue"] / visitors["purchases"]).round(2)
print(visitors["sessions"].sum(), visitors["purchases"].sum(), visitors["revenue"].sum())
visitors

360129 4044 307640.0


,visitor_type,sessions,reached_view,reached_cart,reached_checkout,purchases,revenue,view_rate,cart_rate,checkout_rate,purchase_rate,conversion_rate,revenue_per_session,avg_order_value
0,new,261238,52231,11643,5974,1506,102941.0,0.1999,0.2229,0.5131,0.2521,0.0058,0.39,68.35
1,returning,98891,25083,8690,5132,2538,204699.0,0.2536,0.3464,0.5906,0.4945,0.0257,2.07,80.65


In [4]:
visitors.to_csv(output_dir / "q5_new_vs_returning.csv", index=False)

In [5]:
sql = """
with first_visits as (
    select
        user_pseudo_id,
        case
            when purchased then 'purchased'
            when began_checkout then 'abandoned checkout'
            when added_to_cart then 'abandoned cart'
            when viewed_item then 'viewed only'
            else 'no product view'
        end as first_visit_result
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
    where is_new_visitor
),

later_buyers as (
    select distinct user_pseudo_id
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
    where purchased
    and not is_new_visitor
)
select
    f.first_visit_result,
    count(*) as users,
    countif(l.user_pseudo_id is not null) as bought_later
from first_visits f
left join later_buyers l
    on f.user_pseudo_id = l.user_pseudo_id
group by first_visit_result
order by users desc
"""

returns = client.query(sql).to_dataframe()
returns["bought_later_rate"] = (returns["bought_later"] / returns["users"]).round(4)
print(returns["users"].sum())
returns

261238


,first_visit_result,users,bought_later,bought_later_rate
0,no product view,209007,473,0.0023
1,viewed only,40588,625,0.0154
2,abandoned cart,5669,299,0.0527
3,abandoned checkout,4468,177,0.0396
4,purchased,1506,77,0.0511


In [6]:
returns.to_csv(output_dir/"q5_return_after_first_visit.csv", index=False)